# <font color="#418FDE" size="6.5" uppercase>**Text Dictionary Features**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Convert dictionaries and categorical mappings into numerical feature matrices. 
- Extract text features using counts, n-grams, vocabulary controls, and TF-IDF weighting. 
- Evaluate hashing and sparse text representations for memory, speed, and interpretability tradeoffs. 


## **1. Dictionary Vectorization**

### **1.1. Mapping Dictionaries**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_01.jpg?v=1783835742" width="250">



>* Dictionaries store named attributes for observations.
>* Vectorization maps them into shared feature columns.

>* Numeric values keep their original magnitudes.
>* Categories become separate indicator features.

>* Handle missing fields with consistent feature layout.
>* Feature names preserve meaning for interpretation.



In [ ]:
#@title Python Code - Mapping Dictionaries

# Map dictionaries into aligned numeric features.
# This example uses only core Python tools.
# Missing fields become zeros automatically here.

# Create small records with mixed value types.
records = [
    {"city": "Boston", "member": "gold", "purchases": 3},
    {"city": "Austin", "purchases": 1},
    {"city": "Boston", "member": "silver", "purchases": 2},
]

# Collect feature names from all records.
feature_names = []
for row in records:
    for key, value in row.items():
        name = key if isinstance(value, (int, float)) else f"{key}={value}"
        if name not in feature_names:
            feature_names.append(name)

# Keep a stable column order.
feature_names = sorted(feature_names)
index_map = {name: i for i, name in enumerate(feature_names)}

# Convert each dictionary into one numeric row.
matrix = []
for row in records:
    vector = [0] * len(feature_names)
    for key, value in row.items():
        name = key if isinstance(value, (int, float)) else f"{key}={value}"
        vector[index_map[name]] = value if isinstance(value, (int, float)) else 1
    matrix.append(vector)

# Show the shared feature space.
print("Feature columns:", feature_names)
print("First row vector:", matrix[0])
print("Second row vector:", matrix[1])
print("Matrix shape:", (len(matrix), len(feature_names)))



### **1.2. Feature Matrix Output**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_02.jpg?v=1783835747" width="250">



>* Rows are records; columns are discovered features.
>* Matrices enable consistent numeric comparison for learning.

>* Categories become separate comparable feature columns.
>* Mixed values support analysis through consistent matrices.

>* Matrices have many columns, mostly empty.
>* Named columns aid efficient learning and interpretation.



In [ ]:
#@title Python Code - Feature Matrix Output

# This script builds a simple feature matrix.
# Each dictionary becomes one matrix row.
# Columns show discovered numeric and category features.

# Create tiny records with mixed value types.
records = [
    {"city": "London", "device": "mobile", "visits": 3},
    {"city": "Paris", "device": "desktop", "visits": 1},
    {"city": "London", "device": "desktop", "visits": 2},
]

# Collect feature names in first-seen order.
feature_names = []
for row in records:
    for key, value in row.items():
        name = key if isinstance(value, (int, float)) else f"{key}={value}"
        if name not in feature_names:
            feature_names.append(name)

# Convert dictionaries into matrix rows.
matrix = []
for row in records:
    vector = []
    for name in feature_names:
        if "=" in name:
            key, value = name.split("=", 1)
            vector.append(1 if row.get(key) == value else 0)
        else:
            vector.append(row.get(name, 0))
    matrix.append(vector)

# Show the discovered feature columns.
print("Feature columns:", feature_names)

# Show each observation and its numeric row.
for index, vector in enumerate(matrix, start=1):
    print(f"Row {index}:", vector)

# Confirm the matrix dimensions clearly.
print("Matrix shape:", (len(matrix), len(feature_names)))



### **1.3. Sparse Feature Matrices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_03.jpg?v=1783835744" width="250">



>* Each record fills few possible features.
>* Sparse matrices store selective, informative absences.

>* Sparse matrices save memory in high dimensions.
>* They speed learning on large categorical data.

>* Sparse columns remain interpretable and meaningful.
>* Wide empty matrices are normal, efficient.



In [ ]:
#@title Python Code - Sparse Feature Matrices

# Sparse matrices store mostly zero values.
# Dictionary features often create wide tables.
# This example shows efficient sparse storage.

# Import tools for matrix building.
import numpy as np
from sklearn.feature_extraction import DictVectorizer

# Create small dictionary based records.
records = [
    {"city": "Boston", "device": "mobile", "purchases": 2},
    {"city": "Austin", "device": "desktop", "purchases": 1},
    {"city": "Boston", "device": "desktop", "purchases": 3},
]

# Vectorize records into a sparse matrix.
vectorizer = DictVectorizer(sparse=True)
X_sparse = vectorizer.fit_transform(records)

# Convert once for a tiny readable preview.
X_dense = X_sparse.toarray()
feature_names = vectorizer.get_feature_names_out()

# Count zero and nonzero entries.
rows, cols = X_sparse.shape
nonzero = X_sparse.nnz
zeros = rows * cols - nonzero

# Print a compact teaching summary.
print("Feature names:", list(feature_names))
print("Shape:", X_sparse.shape)
print("Nonzero entries:", nonzero)
print("Zero entries:", zeros)

# Show why sparse storage matters.
print("Dense preview:\n", X_dense)
print("Sparse row 0 indices:", X_sparse[0].indices.tolist())
print("Sparse row 0 values:", X_sparse[0].data.tolist())



## **2. Text Count Features**

### **2.1. Count Vectors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_01.jpg?v=1783835754" width="250">



>* Count vectors turn text into word frequencies.
>* Useful despite ignoring word order and grammar.

>* Count vectors are simple and interpretable.
>* Preprocessing choices strongly shape feature quality.

>* Count vectors lead to richer text features.
>* Sparse counts scale well across documents.



In [ ]:
#@title Python Code - Count Vectors

# Count vectors turn text into simple numeric features.
# This example builds counts from tiny sample documents.
# It also shows vocabulary and sparse matrix structure.

# Import a small text feature tool.
from sklearn.feature_extraction.text import CountVectorizer

# Create a tiny document collection.
documents = [
    "fast delivery and excellent service",
    "damaged package and slow delivery",
    "excellent refund service and fast response",
]

# Build count vectors from text.
vectorizer = CountVectorizer()
count_matrix = vectorizer.fit_transform(documents)

# Get the learned vocabulary terms.
terms = vectorizer.get_feature_names_out()

# Convert sparse counts to dense form.
dense_counts = count_matrix.toarray()

# Check the matrix shape safely.
assert dense_counts.shape == (3, len(terms))

# Print a compact summary.
print("Vocabulary:", list(terms))
print("Matrix shape:", dense_counts.shape)
print("Document 1 counts:", dense_counts[0].tolist())
print("Document 2 counts:", dense_counts[1].tolist())
print("Document 3 counts:", dense_counts[2].tolist())
print("Nonzero entries:", count_matrix.nnz)
print("Sparse storage type:", type(count_matrix).__name__)



### **2.2. Token Counts**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_02.jpg?v=1783835756" width="250">



>* Count tokens to numerically represent text.
>* Repeated words reveal themes across documents.

>* Preprocessing choices shape token counts and meaning.
>* Important tokens vary by task and domain.

>* Counts capture repetition strength beyond binary features.
>* Useful, but biased by length and common words.



In [ ]:
#@title Python Code - Token Counts

# This script demonstrates simple token counting.
# It uses only Python built in tools.
# Small outputs keep the example beginner friendly.

# Create a few short example documents.
documents = [
    "fresh food and friendly staff",
    "fresh fresh salad and tasty soup",
    "friendly service and fresh bread",
]

# Convert text to lowercase word tokens.
tokenized_docs = [doc.lower().split() for doc in documents]

# Build a sorted vocabulary from tokens.
vocabulary = sorted({token for doc in tokenized_docs for token in doc})

# Count each vocabulary token per document.
count_matrix = [
    [doc.count(word) for word in vocabulary]
    for doc in tokenized_docs
]

# Show the learned vocabulary.
print("Vocabulary:", vocabulary)

# Show token counts for each document.
for index, counts in enumerate(count_matrix, start=1):
    pairs = dict(zip(vocabulary, counts))
    print(f"Document {index} counts:", pairs)

# Compare binary presence with true counts.
binary_matrix = [
    [1 if value > 0 else 0 for value in row]
    for row in count_matrix
]

# Show why repeated words matter.
print("Binary for document 2:", dict(zip(vocabulary, binary_matrix[1])))
print("Counts for document 2:", dict(zip(vocabulary, count_matrix[1])))



### **2.3. Word Count Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_03.jpg?v=1783835759" width="250">



>* Word counts turn text into numeric vectors.
>* Repeated words reveal topics, emphasis, and intent.

>* Word counts capture frequency and importance.
>* Interpret counts with length and normalization.

>* Vocabulary choices strongly affect count feature quality.
>* Bag-of-words is useful but ignores order.



In [ ]:
#@title Python Code - Word Count Features

# This script demonstrates simple word count features.
# It uses tiny text data for clarity.
# Results stay short and easy reading.

# Import a small text feature tool.
from sklearn.feature_extraction.text import CountVectorizer

# Create a tiny document collection.
documents = [
    "fresh food and friendly service",
    "late delivery and cold food",
    "fresh fresh salad and delicious food",
]

# Build word count features.
vectorizer = CountVectorizer()
count_matrix = vectorizer.fit_transform(documents)

# Convert sparse counts into a table.
feature_names = vectorizer.get_feature_names_out()
count_table = count_matrix.toarray()

# Check the expected matrix shape.
assert count_table.shape[0] == len(documents)
assert count_table.shape[1] == len(feature_names)

# Show the learned vocabulary.
print("Vocabulary:", list(feature_names))

# Show each document count vector.
for index, row in enumerate(count_table, start=1):
    print(f"Doc {index} counts:", row.tolist())

# Compare one important word frequency.
food_index = list(feature_names).index("food")
food_counts = count_table[:, food_index].tolist()
print("'food' counts by document:", food_counts)



## **3. Sparse TFIDF Pipelines**

### **3.1. Sparse Matrix Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_01.jpg?v=1783835749" width="250">



>* Sparse matrices save memory in text data.
>* They require specialized operations for processing.

>* Sparse TF-IDF saves memory, helps linear models.
>* Some algorithms slow down or need dense data.

>* Sparse TF-IDF is interpretable and explainable.
>* Vocabulary management can be noisy and complex.



In [ ]:
#@title Python Code - Sparse Matrix Tradeoffs

# Sparse matrices save memory for text features.
# This script compares sparse and dense storage.
# It also shows hashing interpretability tradeoffs.

# Import tiny tools for the demonstration.
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer

# Build a tiny text collection.
docs = [
    "fast delivery and friendly service",
    "slow delivery and damaged package",
    "friendly support solved billing error",
    "billing issue and slow response",
]

# Create a sparse TF-IDF matrix.
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_sparse = tfidf.fit_transform(docs)

# Convert once for a size comparison.
X_dense = X_sparse.toarray()
shape_ok = X_sparse.shape[0] == len(docs)

# Count nonzero entries and density.
nonzero = X_sparse.nnz
total = X_sparse.shape[0] * X_sparse.shape[1]
density = nonzero / total if total else 0.0

# Estimate sparse storage bytes.
sparse_bytes = (
    X_sparse.data.nbytes + X_sparse.indices.nbytes + X_sparse.indptr.nbytes
)

# Estimate dense storage bytes.
dense_bytes = X_dense.nbytes
ratio = dense_bytes / sparse_bytes if sparse_bytes else float("inf")

# Build a hashing representation.
hasher = HashingVectorizer(n_features=8, alternate_sign=False, norm=None)
X_hash = hasher.transform(docs)

# Show compact, beginner-friendly results.
print(f"Shape valid: {shape_ok}")
print(f"TF-IDF shape: {X_sparse.shape}")
print(f"Nonzero entries: {nonzero} of {total}")
print(f"Matrix density: {density:.3f}")
print(f"Sparse bytes: {sparse_bytes}")
print(f"Dense bytes: {dense_bytes}")
print(f"Dense/sparse size ratio: {ratio:.1f}x")
print(f"Hashing shape: {X_hash.shape}")
print("TF-IDF columns map to words; hashing columns do not.")



### **3.2. Sparse Matrix Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_02.jpg?v=1783835751" width="250">



>* Sparse matrices save memory in text data.
>* Compressed storage needs specialized algorithm handling.

>* Sparse TF-IDF is fast for linear models.
>* Best choice depends on model and scale.

>* Sparse TF-IDF features are easy to interpret.
>* High dimensions require careful vocabulary and pruning.



In [ ]:
#@title Python Code - Sparse Matrix Tradeoffs

# Sparse matrices save memory for text features.
# This script compares dense and sparse tradeoffs.
# It also shows hashing interpretability limits.

# Import only small allowed tools.
import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer

# Build a tiny repeated corpus.
docs = [
    "refund delayed package arrived late",
    "broken screen refund requested",
    "late delivery and damaged box",
    "great price fast shipping",
] * 40

# Create a sparse TF-IDF matrix.
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_sparse = tfidf.fit_transform(docs)

# Convert once for a memory comparison.
X_dense = X_sparse.toarray()
rows, cols = X_sparse.shape
nnz = X_sparse.nnz

# Estimate sparse storage bytes.
sparse_bytes = (
    X_sparse.data.nbytes + X_sparse.indices.nbytes + X_sparse.indptr.nbytes
)

# Estimate dense storage bytes.
dense_bytes = X_dense.nbytes
sparsity = 1.0 - (nnz / (rows * cols))

# Compare a simple row sum speed.
_ = X_sparse.sum(axis=1)
_ = X_dense.sum(axis=1)

# Build hashed features without vocabulary names.
hash_vec = HashingVectorizer(n_features=16, alternate_sign=False, norm=None)
X_hash = hash_vec.transform(docs)

# Show compact beginner-friendly results.
print(f"documents: {rows}, features: {cols}, nonzeros: {nnz}")
print(f"sparsity: {sparsity:.1%}")
print(f"dense bytes: {dense_bytes:,}")
print(f"sparse bytes: {sparse_bytes:,}")
print(f"dense/sparse ratio: {dense_bytes / sparse_bytes:.1f}x")
print(f"sample named feature: {tfidf.get_feature_names_out()[0]}")
print(f"hash matrix shape: {X_hash.shape}")
print("hashing is compact, but feature names are unavailable")



### **3.3. Sparse Matrix Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_03.jpg?v=1783835753" width="250">



>* Sparse matrices save memory for text features.
>* They scale well but reduce readability.

>* Sparse TF-IDF saves memory and speeds training.
>* Benefits depend on models, operations, and goals.

>* Explicit vocabularies make features easier to explain.
>* Balance interpretability against dimensionality and efficiency.



In [ ]:
#@title Python Code - Sparse Matrix Tradeoffs

# Sparse text matrices save memory efficiently.
# Dense views are easier to inspect.
# Hashing trades names for speed.

import sys
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer

# Build a tiny text collection.
docs = [
    "refund requested for damaged package",
    "urgent refund needed for late delivery",
    "package arrived on time and works",
    "delivery delay caused urgent complaint",
]

# Create an interpretable TF-IDF matrix.
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_tfidf = tfidf.fit_transform(docs)
feature_names = tfidf.get_feature_names_out()

# Create a hashed sparse matrix.
hasher = HashingVectorizer(n_features=8, norm=None, alternate_sign=False)
X_hash = hasher.transform(docs)

# Compare sparse and dense memory roughly.
dense_bytes = X_tfidf.toarray().nbytes
sparse_bytes = (
    X_tfidf.data.nbytes + X_tfidf.indices.nbytes + X_tfidf.indptr.nbytes
)

# Count a simple hash collision estimate.
unique_terms = len(feature_names)
used_buckets = np.unique(X_hash.indices).size
collisions = unique_terms - used_buckets

# Show compact beginner-friendly results.
print("TF-IDF shape:", X_tfidf.shape)
print("Nonzero TF-IDF entries:", X_tfidf.nnz)
print("Dense bytes:", dense_bytes)
print("Sparse bytes:", sparse_bytes)

# Explain interpretability and collisions.
print("First five named features:", list(feature_names[:5]))
print("Hash matrix shape:", X_hash.shape)
print("Used hash buckets:", used_buckets)
print("Estimated collisions:", max(collisions, 0))

# Summarize the main tradeoff.
print("Named sparse features are explainable; hashing is compact but anonymous.")



# <font color="#418FDE" size="6.5" uppercase>**Text Dictionary Features**</font>


In this lecture, you learned to:
- Convert dictionaries and categorical mappings into numerical feature matrices. 
- Extract text features using counts, n-grams, vocabulary controls, and TF-IDF weighting. 
- Evaluate hashing and sparse text representations for memory, speed, and interpretability tradeoffs. 

In the next Lecture (Lecture B), we will go over 'Images Custom Transformers'